# C6 example 4/4: `RestrictedWETensorProduct` with internal spherical harmonics

This notebook keeps the same physical features and hidden architecture, but uses e3nn's 3D spherical harmonics restricted to the planar C6 subgroup of O(3). C6 acts by rotations around z. Thus each input/output 3D vector is still explicitly represented as $(x,y)\in E_1$ plus $z\in A$.

The important API difference is that `RestrictedWETensorProduct.forward_from_points` accepts the geometric 3D point directly. It owns a `RestrictedSphericalHarmonics` evaluator and computes the filter internally:

$$r\xrightarrow{Y_0\oplus\cdots\oplus Y_3}Y(r)\xrightarrow{\mathrm{RestrictedWETP}}\text{features}.$$

The default C6 full bandlimit is $L_{full}=3$, so degrees $l=0,1,2,3$ are included.

In [ ]:
import torch
from we3nn import CyclicGroup, RestrictedSphericalHarmonics, gspaces, nn

torch.manual_seed(7)
torch.set_printoptions(precision=5, sci_mode=False)
G = CyclicGroup(6)
space = gspaces.no_base_space(G)
A = G.trivial_representation
E1 = G.standard_representation
regular = G.regular_representation()
input_type = nn.FieldType(space, 3 * [E1] + 5 * [A])
hidden_type = nn.FieldType(space, 2 * [regular])
output_type = nn.FieldType(space, [E1] + 4 * [A])
spherical = RestrictedSphericalHarmonics(
    G, degrees=None, normalization='component', basis='o3'
)
filter_type = spherical.out_type
print('spherical degrees:', spherical.degrees)
print('restricted filter fields:', [rep.name for rep in filter_type])
print('dimensions:', input_type.size, 'x', filter_type.size, '->', hidden_type.size, '->', output_type.size)

In [ ]:
def pack_input(vectors, scalars):
    xy = vectors[..., :, :2].reshape(*vectors.shape[:-2], 6)
    return torch.cat((xy, vectors[..., :, 2], scalars), dim=-1)

def unpack_input(x):
    xy = x[..., :6].reshape(*x.shape[:-1], 3, 2)
    return torch.cat((xy, x[..., 6:9].unsqueeze(-1)), dim=-1), x[..., 9:11]

def unpack_output(y):
    return torch.cat((y[..., :2], y[..., 2:3]), dim=-1), y[..., 3:6]

def rotate_points(points, element):
    Rxy = E1(element).to(device=points.device, dtype=points.dtype)
    return torch.cat((points[..., :2] @ Rxy.T, points[..., 2:3]), dim=-1)

vectors = torch.tensor([[[1.0, 0.2, -0.4], [-0.3, 0.8, 1.2], [0.5, -0.7, 0.1]]])
scalars = torch.tensor([[0.6, -1.1]])
point = torch.tensor([[0.8, 0.35, -0.2]])
x = input_type.wrap(pack_input(vectors, scalars))
Y = spherical(point)
print('point:', point)
print('internally used restricted spherical harmonics Y_l(r):', Y)
print('harmonic block sizes:', [2*l + 1 for l in spherical.degrees])

## Architecture

The contraction has the same Wigner--Eckart form as notebook 3,

$$z_o=\sum_p w_p(\lVert r\rVert,r_z)(C_p)_{oij}x_iY_j(r),$$

but `forward_from_points` evaluates e3nn spherical harmonics inside the layer. The $C_p$ basis spans the complete finite-C6 Hom space after restriction; it is not limited to parent-O(3) coupling paths. The radial networks use only C6-invariant quantities.

In [ ]:
class RestrictedHarmonicNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.harmonics = spherical
        self.input_layer = nn.RestrictedWETensorProduct(
            input_type, self.harmonics, hidden_type, shared_weights=False
        )
        self.activation = nn.PointActiv(hidden_type, torch.relu)
        self.output_layer = nn.RestrictedWETensorProduct(
            hidden_type, self.harmonics, output_type, shared_weights=False
        )
        self.radial_in = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.input_layer.weight_numel),
        )
        self.radial_out = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.output_layer.weight_numel),
        )

    def forward(self, features, points):
        invariant_geometry = torch.stack(
            (torch.linalg.vector_norm(points, dim=-1), points[..., 2]), dim=-1
        )
        w_in = self.radial_in(invariant_geometry)
        w_out = self.radial_out(invariant_geometry)
        # Spherical filter evaluation happens inside both calls.
        h_pre = self.input_layer.forward_from_points(features, points, w_in)
        h = self.activation(h_pre)
        y = self.output_layer.forward_from_points(h, points, w_out)
        return y, w_in, w_out, h_pre, h

model = RestrictedHarmonicNetwork().eval()
y, w_in, w_out, h_pre, h = model(x, point)
print('input/output reduced-weight counts:', model.input_layer.weight_numel, model.output_layer.weight_numel)
print('hidden before PointActiv:', h_pre.tensor)
print('hidden after  PointActiv:', h.tensor)
print('physical output (vector, scalars):', unpack_output(y.tensor))
kernel_basis = model.input_layer.sample_kernel_basis(point)
print('sampled input-kernel basis shape [batch, paths, out, in]:', tuple(kernel_basis.shape))

## All six simultaneous rotations

The input feature fiber and the 3D point are rotated together. We print the spherical harmonics at every rotated point, all physical inputs and outputs, and the direct equivariance residual.

In [ ]:
errors = []
for k, element in enumerate(G.elements):
    x_k = x.transform_fibers(element)
    point_k = rotate_points(point, element)
    y_k, w_in_k, w_out_k, _, _ = model(x_k, point_k)
    expected_k = y.transform_fibers(element)
    Y_k = spherical(point_k)
    error = (y_k.tensor - expected_k.tensor).abs().max().item()
    errors.append(error)
    torch.testing.assert_close(y_k.tensor, expected_k.tensor, atol=8e-5, rtol=8e-5)
    torch.testing.assert_close(w_in_k, w_in, atol=1e-6, rtol=1e-6)
    torch.testing.assert_close(w_out_k, w_out, atol=1e-6, rtol=1e-6)
    in_vectors_k, in_scalars_k = unpack_input(x_k.tensor)
    out_vector_k, out_scalars_k = unpack_output(y_k.tensor)
    print(f'rotation {k}: angle={60*k:3d} degrees')
    print('  rotated geometry:', point_k[0].tolist())
    print('  spherical Y     :', Y_k[0].tolist())
    print('  input vectors   :', in_vectors_k[0].tolist())
    print('  input scalars   :', in_scalars_k[0].tolist())
    print('  output vector   :', out_vector_k[0].tolist())
    print('  output scalars  :', out_scalars_k[0].tolist())
    print(f'  max equivariance error: {error:.3e}')

print('maximum over all rotations:', max(errors))